<a href="https://www.kaggle.com/code/jeffbutt/tfidf-logreg-baseline?scriptVersionId=322757826" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# TF-IDF + Logistic Regression Baseline

This notebook builds a clean baseline for the Kaggle **LLM Classification Finetuning** competition.

**Project Goal**
Given: 
* `prompt`
* `response_a`
* `response_b`

We will predict 
* `winner_model_a`
* `winner_model_b`
* `winner_tie`

**The Approach** 
1. Load train/test/sample submission files
2. Parse the JSON-like text fields
3. Create 3-class target labels
4. Build comparison text
5. Add A/B swap augmentation
6. Train a TF-IDF + Logistic Regression baseline
7. Validate with multiclass log loss
8. Train a final model on all training data
9. Create `submission.csv`


## 01.  Setup and Config 

In [1]:
import os
import json
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    log_loss,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [2]:
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_colwidth", 500)
pd.set_option("display.max_columns", 100)

In [3]:
DATA_DIR = Path("/kaggle/input/competitions/llm-classification-finetuning")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

In [4]:
print("Using input directory:", DATA_DIR)
print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)
print("Sample submission path:", SAMPLE_SUB_PATH)

Using input directory: /kaggle/input/competitions/llm-classification-finetuning
Train path: /kaggle/input/competitions/llm-classification-finetuning/train.csv
Test path: /kaggle/input/competitions/llm-classification-finetuning/test.csv
Sample submission path: /kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv


## 02. Load Datsaet 

In [5]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

Train shape: (57477, 9)
Test shape: (3, 4)
Sample submission shape: (3, 4)


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain percentage of females on managerial positions?"",""OK, does pineapple belong on a pizza? Relax and give me fun answer.""]","[""The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.\n\nHere are some arguments in favor of and against such policies:\n\n**Arguments in favor:**\n\n1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming fo...","[""As an AI, I don't have personal beliefs or opinions. However, I can tell you that the question of gender quotas in managerial positions is a complex one and positions can vary. Advocates argue that such measures can help correct historical imbalances, promote diversity, and may lead to better decision-making. Critics may argue that positions should be based purely on merit, and that quotas could potentially lead to tokenism or unqualified candidates being promoted. Morality can be subjecti...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage license and marriage certificate?"",""How can I get both of them as quick as possible in California "",""What is the minimal time to get them? 1 day or 10 days?""]","[""A marriage license is a legal document that allows a couple to get married. It is issued by a government agency, such as a county clerk's office or a state government, and is valid for a certain period of time, usually one year. After the marriage has taken place, the couple must obtain a marriage certificate, which is a document that records the marriage and is used to prove that the marriage took place. The marriage certificate is usually issued by the same government agency that issued ...","[""A marriage license and a marriage certificate are two different legal documents that have separate purposes.\n\n1. Marriage License: A marriage license is a legal document that gives a couple permission to get married. It's usually obtained from local government or court officials before the wedding ceremony takes place. The couple is required to meet certain criteria, such as being of a certain age or not being closely related. Once the license is issued, there's often a waiting period be...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call a function?""]","[""Function calling is the process of invoking or executing a function in a programming language. To call a function, you need to use the function's name followed by parentheses \""()\"". Within the parentheses, you can pass arguments if the function requires any.\n\nHere is an example of how to call a function in Python:\n\n```python\n# Define a function\ndef greet(name):\n print(\""Hello, \"" + name + \""!\"")\n\n# Call the function\ngreet(\""John\"")\n```\n\nIn this example, the function \""gree...","[""Function calling is the process of invoking a function in a program. A function is a block of code that performs a specific task and can be reused multiple times in a program. Functions help to organize and modularize code, making it more readable and maintainable.\n\nTo call a function, you simply write the name of the function followed by parentheses () containing any arguments or parameters that the function requires. Here's an example:\n```\nfunction greet(name) {\n console.log(\""Hell...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare category? I want to build a classifier.""]","[""Creating a test set for a very rare category can be challenging, as it may not be possible to collect a sufficient number of examples for the rare category. However, there 

,id,prompt,response_a,response_b
0,136060,"[""I have three oranges today, I ate an orange yesterday. How many oranges do I have?""]","[""You have two oranges today.""]","[""You still have three oranges. Eating an orange yesterday does not affect the number of oranges you have today.""]"
1,211333,"[""You are a mediator in a heated political debate between two opposing parties. Mr Reddy is very hung up on semantic definitions of sex and gender, and believes that women are adult human females. Meanwhile Ms Blue is extremely fluid with definitions and does not care about truth. He (Ms blue uses he\/him pronouns) insists that anybody can be any gender, gametes don't mean anything, and that men can get pregnant. You, Mr Goddy are tasked with helping them both find a middle ground.""]","[""Thank you for sharing the details of the situation. As a mediator, I understand the importance of finding a middle ground that both parties can agree upon.\n\nIt is important to recognize that language and definitions can be complex and can vary depending on cultural and societal norms. At the same time, it is also important to acknowledge the biological differences between males and females.\n\nOne possible approach to finding a middle ground could be to focus on respecting individuals' s...","[""Mr Reddy and Ms Blue both have valid points in their arguments. On one hand, Mr Reddy is correct that women are adult human females, as defined by their biological and physical characteristics. However, Ms Blue is right that gender is a more fluid concept and that people can identify with any gender they choose. A compromise could be that gender is defined by an individual's self-identification, while sex is defined by their biological characteristics.""]"
2,1233961,"[""How to initialize the classification head when I do transfer learning. For example, I have a pre-trained vision transformer on ImageNet, and now I want to finetune it on StanfordCars"",""I want to do full finetuning""]","[""When you want to initialize the classification head for transfer learning, you can follow these steps:\n\n 1. Load the pre-trained model: First, load the pre-trained vision transformer model on ImageNet. You can use popular deep learning libraries like PyTorch or TensorFlow to load the model.\n 2. Freeze the pre-trained layers: Next, you need to freeze the pre-trained layers of the model so that they don't get updated during the training process. This is because the pre-trained layers ha...","[""To initialize the classification head when performing transfer learning, follow these steps:\n\n1. Load the pre-trained vision transformer model that was trained on ImageNet.\n2. Replace the existing classification head with a new one suitable for the StanfordCars dataset.\n3. Initialize the weights of the new classification head with random values or pre-trained weights if available.\n4. Freeze the weights of the vision transformer backbone to prevent them from being updated during traini..."


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.333333,0.333333,0.333333
1,211333,0.333333,0.333333,0.333333
2,1233961,0.333333,0.333333,0.333333


## 03. Inspect Columns

Important note: `model_a` and `model_b` appear in train, but not in test.  
This baseline does **not** use model names.

In [6]:
print("Train columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

Train columns:
['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']

Test columns:
['id', 'prompt', 'response_a', 'response_b']


In [7]:
required_train_cols = {
    "id",
    "prompt",
    "response_a",
    "response_b",
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
}

required_test_cols = {
    "id",
    "prompt",
    "response_a",
    "response_b",
}

missing_train = required_train_cols - set(train.columns)
missing_test = required_test_cols - set(test.columns)

if missing_train:
    raise ValueError(f"Missing required train columns: {missing_train}")

if missing_test:
    raise ValueError(f"Missing required test columns: {missing_test}")

print("\nColumn checks passed.")


Column checks passed.


## 04. Parse Text Fields 
The text fields are often stored as JSON-like lists, for example:

```text
["First turn", "Second turn"]
```

We safely parse these and join list items with newlines.

In [8]:
def parse_text_field(x):
    """
    Safely parse text fields that may be JSON-encoded lists.

    Examples:
    - '["hello", "world"]' -> 'hello\nworld'
    - 'plain text' -> 'plain text'
    - NaN -> ''
    """
    if pd.isna(x):
        return ""

    if not isinstance(x, str):
        return str(x)

    try:
        value = json.loads(x)

        if isinstance(value, list):
            return "\n".join("" if item is None else str(item) for item in value)

        if value is None:
            return ""

        return str(value)

    except Exception:
        return x


for col in ["prompt", "response_a", "response_b"]:
    train[f"{col}_parsed"] = train[col].apply(parse_text_field)
    test[f"{col}_parsed"] = test[col].apply(parse_text_field)

display(train[[
    "prompt_parsed",
    "response_a_parsed",
    "response_b_parsed",
]].head(2))

,prompt_parsed,response_a_parsed,response_b_parsed
0,"Is it morally right to try to have a certain percentage of females on managerial positions?\nOK, does pineapple belong on a pizza? Relax and give me fun answer.","The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.\n\nHere are some arguments in favor of and against such policies:\n\n**Arguments in favor:**\n\n1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for ...","As an AI, I don't have personal beliefs or opinions. However, I can tell you that the question of gender quotas in managerial positions is a complex one and positions can vary. Advocates argue that such measures can help correct historical imbalances, promote diversity, and may lead to better decision-making. Critics may argue that positions should be based purely on merit, and that quotas could potentially lead to tokenism or unqualified candidates being promoted. Morality can be subjective..."
1,What is the difference between marriage license and marriage certificate?\nHow can I get both of them as quick as possible in California \nWhat is the minimal time to get them? 1 day or 10 days?,"A marriage license is a legal document that allows a couple to get married. It is issued by a government agency, such as a county clerk's office or a state government, and is valid for a certain period of time, usually one year. After the marriage has taken place, the couple must obtain a marriage certificate, which is a document that records the marriage and is used to prove that the marriage took place. The marriage certificate is usually issued by the same government agency that issued th...","A marriage license and a marriage certificate are two different legal documents that have separate purposes.\n\n1. Marriage License: A marriage license is a legal document that gives a couple permission to get married. It's usually obtained from local government or court officials before the wedding ceremony takes place. The couple is required to meet certain criteria, such as being of a certain age or not being closely related. Once the license is issued, there's often a waiting period befo..."


## 05. Create Target Labels 

Label mapping:

```text
0 = winner_model_a
1 = winner_model_b
2 = winner_tie
```

In [9]:
target_cols = ["winner_model_a", "winner_model_b", "winner_tie"]

train["target"] = train[target_cols].values.argmax(axis=1)

label_names = {
    0: "winner_model_a",
    1: "winner_model_b",
    2: "winner_tie",
}

print("Target distribution:")
target_dist = train["target"].value_counts(normalize=True).sort_index()
for idx, value in target_dist.items():
    print(f"{idx} ({label_names[idx]}): {value:.4f}")

display(train[target_cols + ["target"]].head())

Target distribution:
0 (winner_model_a): 0.3491
1 (winner_model_b): 0.3419
2 (winner_tie): 0.3090


,winner_model_a,winner_model_b,winner_tie,target
0,1,0,0,0
1,0,1,0,1
2,0,0,1,2
3,1,0,0,0
4,0,1,0,1


## 06. Build Comparison Text

The model sees the prompt and both candidate responses together.


In [10]:
def build_comparison_text(df):
    """
    Build text input for pairwise response comparison.
    """
    return (
        "PROMPT:\n" + df["prompt_parsed"].fillna("") +
        "\n\nRESPONSE A:\n" + df["response_a_parsed"].fillna("") +
        "\n\nRESPONSE B:\n" + df["response_b_parsed"].fillna("")
    )


train["text"] = build_comparison_text(train)
test["text"] = build_comparison_text(test)

print(train["text"].iloc[0][:2000])

PROMPT:
Is it morally right to try to have a certain percentage of females on managerial positions?
OK, does pineapple belong on a pizza? Relax and give me fun answer.

RESPONSE A:
The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.

Here are some arguments in favor of and against such policies:

**Arguments in favor:**

1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.

2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and society as a whole.

3. **Equality of Oppor

## 07. A/B Swap Augmentation

For each training row, create a swapped version:

- `response_a` and `response_b` are swapped
- A-win becomes B-win
- B-win becomes A-win
- Tie remains tie

This reduces position bias and teaches symmetry.

In [11]:
def create_swapped_training_data(df):
    """
    Create augmented rows where response_a and response_b are swapped.

    Original labels:
    0 = A wins
    1 = B wins
    2 = tie

    Swapped labels:
    0 -> 1
    1 -> 0
    2 -> 2
    """
    swapped = df.copy()

    swapped["response_a_parsed"] = df["response_b_parsed"].values
    swapped["response_b_parsed"] = df["response_a_parsed"].values

    swapped["text"] = build_comparison_text(swapped)

    swapped["target"] = df["target"].map({
        0: 1,
        1: 0,
        2: 2,
    }).astype(int).values

    return swapped


swapped_preview = create_swapped_training_data(train.head(3))

print("Original target:", train.loc[:2, "target"].tolist())
print("Swapped target:", swapped_preview["target"].tolist())
display(swapped_preview[["text", "target"]].head(1))

Original target: [0, 1, 2]
Swapped target: [1, 0, 2]


,text,target
0,"PROMPT:\nIs it morally right to try to have a certain percentage of females on managerial positions?\nOK, does pineapple belong on a pizza? Relax and give me fun answer.\n\nRESPONSE A:\nAs an AI, I don't have personal beliefs or opinions. However, I can tell you that the question of gender quotas in managerial positions is a complex one and positions can vary. Advocates argue that such measures can help correct historical imbalances, promote diversity, and may lead to better decision-making....",1


## 08. Train/Validation Split

We split the original rows first, then augment only the training split.

This avoids leaking swapped versions of validation rows into training.

In [12]:
train_base, valid_base = train_test_split(
    train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=train["target"],
)

train_base_swapped = create_swapped_training_data(train_base)

train_model_df = pd.concat(
    [
        train_base[["text", "target"]],
        train_base_swapped[["text", "target"]],
    ],
    axis=0,
    ignore_index=True,
)

valid_model_df = valid_base[["text", "target"]].copy()

X_train = train_model_df["text"].values
y_train = train_model_df["target"].values

X_valid = valid_model_df["text"].values
y_valid = valid_model_df["target"].values

print("Original train rows:", train.shape[0])
print("Train split rows before augmentation:", train_base.shape[0])
print("Training rows after augmentation:", train_model_df.shape[0])
print("Validation rows:", valid_model_df.shape[0])

print("\nTraining target distribution after augmentation:")
print(pd.Series(y_train).value_counts(normalize=True).sort_index())

print("\nValidation target distribution:")
print(pd.Series(y_valid).value_counts(normalize=True).sort_index())

Original train rows: 57477
Train split rows before augmentation: 45981
Training rows after augmentation: 91962
Validation rows: 11496

Training target distribution after augmentation:
0    0.345491
1    0.345491
2    0.309019
Name: proportion, dtype: float64

Validation target distribution:
0    0.349078
1    0.341945
2    0.308977
Name: proportion, dtype: float64


## 09. Build TF-IDF + Logistic Regression Model

This is a strong, simple baseline.

Notes:

- Word unigrams and bigrams
- Up to 300,000 features
- Multinomial logistic regression
- `saga` solver for sparse high-dimensional text features

In [13]:
def make_tfidf_logreg_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="word",
            ngram_range=(1, 2),
            max_features=300_000,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            dtype=np.float32,
        )),
        ("clf", LogisticRegression(
            C=2.0,
            solver="saga",
            multi_class="multinomial",
            max_iter=1000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=1,
        )),
    ])


model = make_tfidf_logreg_model()
model

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(dtype=<class 'numpy.float32'>, max_df=0.95,
                                 max_features=300000, min_df=2,
                                 ngram_range=(1, 2), strip_accents='unicode',
                                 sublinear_tf=True)),
                ('clf',
                 LogisticRegression(C=2.0, max_iter=1000,
                                    multi_class='multinomial', n_jobs=-1,
                                    random_state=42, solver='saga',
                                    verbose=1))])

## 10. Train Validation Model

In [14]:
model.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


convergence after 32 epochs took 71 seconds


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(dtype=<class 'numpy.float32'>, max_df=0.95,
                                 max_features=300000, min_df=2,
                                 ngram_range=(1, 2), strip_accents='unicode',
                                 sublinear_tf=True)),
                ('clf',
                 LogisticRegression(C=2.0, max_iter=1000,
                                    multi_class='multinomial', n_jobs=-1,
                                    random_state=42, solver='saga',
                                    verbose=1))])

## 11. Validate 

In [15]:
valid_probs = model.predict_proba(X_valid)
valid_preds = valid_probs.argmax(axis=1)

valid_logloss = log_loss(y_valid, valid_probs, labels=[0, 1, 2])
valid_acc = accuracy_score(y_valid, valid_preds)

print(f"Validation Log Loss: {valid_logloss:.6f}")
print(f"Validation Accuracy: {valid_acc:.6f}")

print("\nClassification Report:")
print(classification_report(
    y_valid,
    valid_preds,
    target_names=["winner_model_a", "winner_model_b", "winner_tie"],
    digits=4,
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, valid_preds))

Validation Log Loss: 1.112051
Validation Accuracy: 0.386743

Classification Report:
                precision    recall  f1-score   support

winner_model_a     0.3806    0.3750    0.3778      4013
winner_model_b     0.3809    0.3638    0.3722      3931
    winner_tie     0.3989    0.4254    0.4117      3552

      accuracy                         0.3867     11496
     macro avg     0.3868    0.3881    0.3872     11496
  weighted avg     0.3864    0.3867    0.3864     11496


Confusion Matrix:
[[1505 1358 1150]
 [1374 1430 1127]
 [1075  966 1511]]


## 12. Probability Diagnostics 

In [16]:
prob_df = pd.DataFrame(
    valid_probs,
    columns=["winner_model_a", "winner_model_b", "winner_tie"],
)

print("Validation probability summary:")
display(prob_df.describe())

print("Average predicted probabilities:")
display(prob_df.mean())

print("Actual validation class distribution:")
actual_valid_dist = pd.Series(y_valid).value_counts(normalize=True).sort_index()
display(actual_valid_dist.rename(index=label_names))

Validation probability summary:


,winner_model_a,winner_model_b,winner_tie
count,11496.000000,11496.000000,11496.000000
mean,0.353347,0.353828,0.292825
std,0.088677,0.088781,0.174724
min,0.007442,0.006515,0.017480
25%,0.302255,0.303565,0.158287
50%,0.372074,0.371891,0.254842
75%,0.420904,0.421065,0.391633
max,0.531368,0.638349,0.986043


Average predicted probabilities:


winner_model_a    0.353346
winner_model_b    0.353828
winner_tie        0.292826
dtype: float32

Actual validation class distribution:


winner_model_a    0.349078
winner_model_b    0.341945
winner_tie        0.308977
Name: proportion, dtype: float64

## 13. Inspect Top Features

In [17]:
feature_names = model.named_steps["tfidf"].get_feature_names_out()
coef = model.named_steps["clf"].coef_

for class_idx, class_name in label_names.items():
    print("\n" + "=" * 90)
    print(f"Top features for: {class_name}")
    print("=" * 90)

    top_idx = np.argsort(coef[class_idx])[-25:][::-1]

    for idx in top_idx:
        print(f"{feature_names[idx]:50s} {coef[class_idx][idx]:.5f}")


Top features for: winner_model_a
do response                                        2.33605
sister response                                    2.20867
questions response                                 2.10565
and                                                1.81036
asking response                                    1.53182
bricks response                                    1.43961
how are                                            1.37557
country response                                   1.33196
to                                                 1.30254
issues response                                    1.24026
down the                                           1.20478
helfen response                                    1.14614
however some                                       1.13505
one sister                                         1.09487
prompt explain                                     1.07984
steel response                                     1.07571
hi there              

## 14. Train Final Model on Full Training Data

In [18]:
# Clear validation model to reduce memory pressure
del model
gc.collect()

full_swapped = create_swapped_training_data(train)

full_train_df = pd.concat(
    [
        train[["text", "target"]],
        full_swapped[["text", "target"]],
    ],
    axis=0,
    ignore_index=True,
)

X_full = full_train_df["text"].values
y_full = full_train_df["target"].values

print("Full training rows before augmentation:", train.shape[0])
print("Full training rows after augmentation:", full_train_df.shape[0])
print("\nFull augmented target distribution:")
print(pd.Series(y_full).value_counts(normalize=True).sort_index())

final_model = make_tfidf_logreg_model()
final_model.fit(X_full, y_full)

Full training rows before augmentation: 57477
Full training rows after augmentation: 114954

Full augmented target distribution:
0    0.345495
1    0.345495
2    0.309011
Name: proportion, dtype: float64


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


convergence after 32 epochs took 90 seconds


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(dtype=<class 'numpy.float32'>, max_df=0.95,
                                 max_features=300000, min_df=2,
                                 ngram_range=(1, 2), strip_accents='unicode',
                                 sublinear_tf=True)),
                ('clf',
                 LogisticRegression(C=2.0, max_iter=1000,
                                    multi_class='multinomial', n_jobs=-1,
                                    random_state=42, solver='saga',
                                    verbose=1))])

## 15. Predict Test Set

In [19]:
test_probs = final_model.predict_proba(test["text"].values)

print("Test probability shape:", test_probs.shape)
print(test_probs[:5])

Test probability shape: (3, 3)
[[0.3392255  0.32630578 0.33446875]
 [0.39981386 0.372017   0.22816911]
 [0.47239915 0.45381278 0.07378807]]


## 16. Clip and Normalize Probabilities

This protects against extreme probabilities hurting log loss.

In [20]:
def clip_and_normalize(preds, eps=1e-5):
    preds = np.clip(preds, eps, 1 - eps)
    preds = preds / preds.sum(axis=1, keepdims=True)
    return preds


test_probs = clip_and_normalize(test_probs)

print("First rows after clipping/normalization:")
print(test_probs[:5])

print("\nRow sums:")
print(test_probs.sum(axis=1)[:5])


First rows after clipping/normalization:
[[0.3392255  0.32630578 0.33446875]
 [0.39981386 0.372017   0.22816911]
 [0.47239915 0.45381278 0.07378807]]

Row sums:
[1. 1. 1.]


## 17. Create and Save Submission

The file must be named:

```text
submission.csv
```

In [21]:
submission = pd.DataFrame({
    "id": test["id"],
    "winner_model_a": test_probs[:, 0],
    "winner_model_b": test_probs[:, 1],
    "winner_tie": test_probs[:, 2],
})

expected_cols = ["id", "winner_model_a", "winner_model_b", "winner_tie"]
submission = submission[expected_cols]

display(submission.head())

print("Submission shape:", submission.shape)
print("\nProbability row sums:")
display(submission[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1).describe())


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.339226,0.326306,0.334469
1,211333,0.399814,0.372017,0.228169
2,1233961,0.472399,0.453813,0.073788


Submission shape: (3, 4)

Probability row sums:


count    3.0
mean     1.0
std      0.0
min      1.0
25%      1.0
50%      1.0
75%      1.0
max      1.0
dtype: float64

In [22]:
submission.to_csv("submission.csv", index=False)

print("Saved submission.csv")
display(pd.read_csv("submission.csv").head())


Saved submission.csv


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.339226,0.326306,0.334469
1,211333,0.399814,0.372017,0.228169
2,1233961,0.472399,0.453813,0.073788
